In [ ]:
# Install Python libraries
! pip3 install numpy==1.23.5
! pip3 install pandas==2.0.1
! pip3 install scanpy==1.9.3
! pip3 install bokeh==3.0.3
! pip3 install selenium==4.33.0

In [ ]:
rscript0 = f"""################################# Install R libraries
## =========================
## 0. 必須: パッケージ管理用
## =========================
if (!requireNamespace("BiocManager", quietly = TRUE)) {
  install.packages("BiocManager")
}

if (!requireNamespace("remotes", quietly = TRUE)) {
  install.packages("remotes")
}

## =========================
## 1. CRAN packages
## =========================
install.packages(c(
  "ggplot2",
  "dplyr",
  "future",
  "patchwork",
  "stringr",
  "Seurat"
))

## =========================
## 2. Bioconductor packages
## =========================
BiocManager::install(c(
  "GenomicRanges",
  "GenomeInfoDb",
  "EnsDb.Mmusculus.v79"
), ask = FALSE, update = FALSE)

## =========================
## 3. Signac
## =========================
install.packages("Signac")

## =========================
## 4. monocle3
## =========================
if (!requireNamespace("monocle3", quietly = TRUE)) {
  remotes::install_github("cole-trapnell-lab/monocle3")
}

## =========================
## 5. SeuratWrappers
## =========================
if (!requireNamespace("SeuratWrappers", quietly = TRUE)) {
  remotes::install_github("satijalab/seurat-wrappers")
}"""

f = open(f'install_libraries.r', 'w')
f.write(rscript0)
f.close()

!Rscript install_libraries.r

# 1. Libraries and settings

In [1]:
# Library
import warnings
warnings.simplefilter('ignore')

import os
import pickle
import numpy as np
import pandas as pd
import scanpy as sc

from bokeh.io import output_notebook, show
from bokeh.models import ColumnDataSource, HoverTool, LinearColorMapper
from bokeh.plotting import figure
from bokeh.models import SaveTool
from bokeh.io import export_png

In [2]:
# scanpy settings
sc.settings.verbosity = 3
# sc.logging.print_versions()
results_file = './write/pbmc3k.h5ad'

In [3]:
# Make directories
os.makedirs('Fig', exist_ok=True)
os.makedirs('Fig/Fig1(b)', exist_ok=True)

In [4]:
# Figure number definitions
Fig1b_GEX_ALL = f'Fig/Figure1(b)_GEX_ALL'

Fig1b_GEX_PLN = f'Fig/Figure1(b)_GEX_PLN'
Fig1b_GEX_MLN = f'Fig/Figure1(b)_GEX_MLN'
Fig1b_GEX_PP = f'Fig/Figure1(b)_GEX_PP'

Fig1b_ATAC_PLN = f'Fig/Figure1(b)_ATAC_PLN'
Fig1b_ATAC_MLN = f'Fig/Figure1(b)_ATAC_MLN'
Fig1b_ATAC_PP = f'Fig/Figure1(b)_ATAC_PP'
Fig1b_ATAC_ALL = f'Fig/Figure1(b)_ATAC_ALL'

Fig1b_WNN_PLN = f'Fig/Figure1(b)_WNN_PLN'
Fig1b_WNN_MLN = f'Fig/Figure1(b)_WNN_MLN'
Fig1b_WNN_PP = f'Fig/Figure1(b)_WNN_PP'
Fig1b_WNN_ALL = f'Fig/Figure1(b)_WNN_ALL'

FigS5 = f'Fig/Supplementary_Figure5'
FigS7 = f'Fig/Supplementary_Figure7'
FigS8sub = f'Fig/Supplementary_Figure8(sub1)'
FigS8a = f'Fig/Supplementary_Figure8(a)'
FigS8b = f'Fig/Supplementary_Figure8(b)'
FigS9 = f'Fig/Supplementary_Figure9'

# 2. Prefiltering

In [5]:
def prefiltering(
    tissue,
    positive_barcords=None,
    min_genes=1000,
    min_genes_second=None,
    max_genes=6000,
    min_cells=3,
    max_percent_mito=0.15
):
    # Loading data
    adata = sc.read_10x_mtx(f'{tissue}/filtered_feature_bc_matrix/', var_names='gene_symbols', cache=True)

    # Filtering by cell count and gene count
    sc.pp.filter_cells(adata, min_genes=min_genes)
    sc.pp.filter_genes(adata, min_cells=min_cells)
    if min_genes_second is not None:
        sc.pp.filter_cells(adata, min_genes=min_genes_second)
    adata = adata[adata.obs['n_genes'] < max_genes, :]

    # Filter cells based on mitochondrial gene expression levels
    mito_genes = adata.var_names.str.startswith('mt-')
    adata.obs['percent_mito'] = np.sum(adata[:, mito_genes].X, axis=1).A1 / np.sum(adata.X, axis=1).A1
    adata = adata[adata.obs['percent_mito'] < max_percent_mito, :]

    # Shape of adata
    print(f'Shape of adata after prefiltering: {adata.shape}')

    # Filter cells annotated as non-vascular endothelial cells
    if positive_barcords is not None:
        b = [i.rstrip() for i in open(f"{tissue}/filtered_feature_bc_matrix/{positive_barcords}").readlines()]
        cells = list(adata.obs.index)
        cells = [i for i in cells if i in b]
        adata = adata[cells,:]

        # Shape of adata
        print(f'Shape of adata after excluding non-vascular endothelial cells: {adata.shape}')

    return adata


def save_matrix(
    tissue,
    adata,
    out_barcodes_fn='barcodes_gex.v2.rev.tsv',
    out_features_fn='features_gex.v2.rev.tsv',
    out_matrix_fn='matrix_gex.v2.rev.mtx'
):
    # Saving barcodes data
    if out_barcodes_fn is not None:
        cells = list(adata.obs.index)
        f = open(f"{tissue}/filtered_feature_bc_matrix/{out_barcodes_fn}", "w")
        f.writelines([i + "\n" for i in cells])
        f.close()

    # Saving feature data
    if out_features_fn is not None:
        features = open(f"{tissue}/filtered_feature_bc_matrix/features.tsv").readlines()
        features_subset = []
        for g in list(adata.var.gene_ids):
            for f in features:
                name = f.split("\t")[0]
                if g == name:
                    features_subset.append(f)
                    break
        f = open(f"{tissue}/filtered_feature_bc_matrix/{out_features_fn}", "w")
        f.writelines(features_subset)
        f.close()

    # Saving matrix data
    if out_matrix_fn is not None:
        matrix = []
        for i in range(adata.X.shape[0]):
            rr = adata.X[i,:].toarray()
            li = [str(j+1)+" "+str(i+1)+" "+str(int(rr[0,j]))+"\n" for j in range(rr.shape[1]) if rr[0,j] != 0]
            matrix.extend(li)
        header = [
            '%%MatrixMarket matrix coordinate integer general\n',
            '%metadata_json: {"software_version": "cellranger-arc-2.0.0", "format_version": 2}\n',
            str(len(features_subset)) + " " + str(len(cells)) + " " + str(len(matrix)) + "\n"
        ]
        f = open(f"{tissue}/filtered_feature_bc_matrix/{out_matrix_fn}", "w")
        f.writelines(header)
        f.writelines(matrix)
        f.close()

In [139]:
# PLN
adata_PLN = prefiltering(
    'PLN',
    min_genes=200,
    max_genes=6000,
    min_cells=3,
    max_percent_mito=0.15
)
save_matrix(
    'PLN',
    adata_PLN,
    out_barcodes_fn='barcodes_gex_subset_final.rev.tsv',
    out_features_fn='features_gex_subset_final.rev.tsv',
    out_matrix_fn='matrix_gex_subset_final.rev.mtx'
)

# MLN
adata_MLN = prefiltering(
    'MLN',
    min_genes=200,
    max_genes=6000,
    min_cells=3,
    max_percent_mito=0.15
)
save_matrix(
    'MLN',
    adata_MLN,
    out_barcodes_fn='barcodes_gex_subset_final.rev.tsv',
    out_features_fn='features_gex_subset_final.rev.tsv',
    out_matrix_fn='matrix_gex_subset_final.rev.mtx'
)

# PP
adata_PP = prefiltering(
    'PP',
    min_genes=200,
    min_genes_second=750,
    max_genes=6000,
    min_cells=3,
    max_percent_mito=0.15
)
save_matrix(
    'PP',
    adata_PP,
    out_barcodes_fn='barcodes_gex_subset_final.rev.tsv',
    out_features_fn='features_gex_subset_final.rev.tsv',
    out_matrix_fn='matrix_gex_subset_final.rev.mtx'
)

... reading from cache file cache/PLN-filtered_feature_bc_matrix-matrix.h5ad
filtered out 64 cells that have less than 200 genes expressed
filtered out 11577 genes that are detected in less than 3 cells
Shape of adata after prefiltering: (7387, 20708)
... reading from cache file cache/MLN-filtered_feature_bc_matrix-matrix.h5ad
filtered out 158 cells that have less than 200 genes expressed
filtered out 11972 genes that are detected in less than 3 cells
Shape of adata after prefiltering: (8703, 20313)
... reading from cache file cache/PP-filtered_feature_bc_matrix-matrix.h5ad
filtered out 663 cells that have less than 200 genes expressed
filtered out 11717 genes that are detected in less than 3 cells
filtered out 2596 cells that have less than 750 genes expressed
Shape of adata after prefiltering: (7985, 20568)


# 3. Clustering

In [7]:
rscript1 = f"""################################
# This is an R script.
# --------------------------------------------------------
# Filename: clustering_cells.r
# --------------------------------------------------------
# How to run
# $ Rscript clustering_cells.r PLN 7 a
# $ Rscript clustering_cells.r MLN 8 b
# $ Rscript clustering_cells.r PP 8 c
# --------------------------------------------------------
# outputs
# - {FigS5}(a-c).pdf
# - {FigS7}(a-c).pdf
# - {{tissue}}/filtered_feature_bc_matrix/cluster_def_by_monocle_final.rev.csv
# - {{tissue}}/filtered_feature_bc_matrix/coordinate_by_monocle_final.rev.csv
################################
clustering_cells <- function(tissue, k, tag){{
    # Library
    library(monocle3)
    library(ggplot2)
    library(dplyr)

    # Loading data
    cds <- load_mm_data(
        mat_path = sprintf("%s/filtered_feature_bc_matrix/matrix_gex_subset_final.rev.mtx", tissue),
        feature_anno_path = sprintf("%s/filtered_feature_bc_matrix/features_gex_subset_final.rev.tsv", tissue),
        cell_anno_path = sprintf("%s/filtered_feature_bc_matrix/barcodes_gex_subset_final.rev.tsv", tissue)
    )

    # Preprocessing
    names(rowData(cds)) <- "gene_short_name"
    cds <- preprocess_cds(cds, num_dim = 20)

    # Dimension reduction
    cds <- reduce_dimension(cds)

    # Checking marker gene expression
    pdf(sprintf("{FigS7}(%s).pdf", tag))
    print(plot_cells(cds, genes=c("Bmx", "Depp1", "Gja4", "Gja5", "Gkn3", "Sox17")) + ggtitle("Art"))
    print(plot_cells(cds, genes=c("Atf3", "Cxcl1", "Egr1", "Fos", "Fosb", "Jun", "Junb", "Jund", "Nfkbia", "Nr4a1", "Sgk1")) + ggtitle("CapEC2"))
    print(plot_cells(cds, genes=c("Col4a1", "Col4a2", "Hlx", "Id1", "Id3", "Igfbp3")) + ggtitle("CapEC1"))
    print(plot_cells(cds, genes=c("Cxcl9", "Cxcl10", "Irf7", "Isg15", "Lgals9")) + ggtitle("CapIfn"))
    print(plot_cells(cds, genes=c("Angpt2", "Apln", "Esm1", "Mcam", "Nid2", "Pdgfb", "Pgf", "Vim")) + ggtitle("CRP"))
    print(plot_cells(cds, genes=c("Apoe", "St3gal6")) + ggtitle("TrEC"))
    print(plot_cells(cds, genes=c("Ccl21a", "Chst4", "Glycam1", "Ttn", "Gcnt1", "Fut7", "Nkx2-3", "fr2")) + ggtitle("HEC"))
    print(plot_cells(cds, genes=c("Ackr1", "Icam1", "Nr2f2", "Sele", "Selp", "Vcam1", "Vwf")) + ggtitle("Vn"))
    dev.off()

    # Clustering and visualization
    cds <- cluster_cells(cds, k=k)
    pdf(sprintf("{FigS5}(%s).pdf", tag))
    print(plot_cells(cds))
    dev.off()

    # Saving clustering results
    cnames <- names(clusters(cds))
    cvalues <- as.numeric(as.character(clusters(cds)))
    cmat <- cbind(cnames, cvalues)
    write.csv(
        x=cmat,
        sprintf("%s/filtered_feature_bc_matrix/cluster_def_by_monocle_final.rev.csv", tissue),
        row.names=F,
        quote=F
    )

    # Saving coordinates
    write.csv(
        x=reducedDims(cds)[["UMAP"]],
        sprintf("%s/filtered_feature_bc_matrix/coordinate_by_monocle_final.rev.csv", tissue),
        quote=F
    )
}}

tissue = commandArgs(trailingOnly=TRUE)[1]
k = as.numeric(commandArgs(trailingOnly=TRUE)[2])
tag = commandArgs(trailingOnly=TRUE)[3]

clustering_cells(tissue, k, tag)"""

f = open(f'clustering_cells.r', 'w')
f.write(rscript1)
f.close()

In [20]:
!Rscript clustering_cells.r PLN 7 a
!Rscript clustering_cells.r MLN 8 b
!Rscript clustering_cells.r PP 8 c

 要求されたパッケージ Biobase をロード中です 
 要求されたパッケージ BiocGenerics をロード中です 

 次のパッケージを付け加えます: ‘BiocGenerics’ 

 以下のオブジェクトは ‘package:stats’ からマスクされています:

    IQR, mad, sd, var, xtabs

 以下のオブジェクトは ‘package:base’ からマスクされています:

    anyDuplicated, append, as.data.frame, basename, cbind, colnames,
    dirname, do.call, duplicated, eval, evalq, Filter, Find, get, grep,
    grepl, intersect, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, setdiff, sort, table, tapply,
    union, unique, unsplit, which.max, which.min

Welcome to Bioconductor

    Vignettes contain introductory material; view with
    'browseVignettes()'. To cite Bioconductor, see
    'citation("Biobase")', and for packages 'citation("pkgname")'.

 要求されたパッケージ SingleCellExperiment をロード中です 
 要求されたパッケージ SummarizedExperiment をロード中です 
 要求されたパッケージ MatrixGenerics をロード中です 
 要求されたパッケージ matrixStats をロード中です 

 次のパッケージを付け加えます: ‘matrixStats’ 

 以下のオブジェクト

# 4. Assign common cluster numbers across three tissues

 Update the cluster numbers in the cluster definition file generated in the previous section to ensure that clusters sharing the same number represent the same cluster across tissues.

In [125]:
# Rules for reassigning cluster numbers
def convert_cluster_number(tissue, num):
    if tissue == 'PLN':
        if num == 6: # Art
            return 1
        elif num == 4: # CapEC2
            return 2
        elif num == 2: # CapEC2
            return 3
        elif num == 5: # CapEC2
            return 4
        elif num == 1: # CapEC1
            return 5
        elif num == 8: # CRP
            return 6
        elif num == 7: # TrEC
            return 7
        elif num == 3: # HEC
            return 8
        elif num == 9: # Vn
            return 9
        else:
            return -1
    elif tissue == 'MLN':
        if num == 3: # Art
            return 1
        elif num == 1: # CapEC2
            return 2
        elif num == 2: # CapEC2
            return 3
        elif num == 4: # CapEC2
            return 4
        elif num == 6: # CapEC1
            return 5
        elif num == 7: # CRP
            return 6
        # Note: TrEC cells are absent in MLNs
        elif num == 5: # HEC
            return 8
        elif num == 9: # Vn
            return 9
        elif num == 8: # Others
            return 10
        else:
            return -1
    elif tissue == 'PP':
        if num == 1: # Art
            return 1
        elif num == 2:# CapEC2
            return 2
        elif num == 3: # CapEC2
            return 3
        elif num == 8: # CapEC2
            return 4
        elif num == 7: # CapEC1
            return 5
        elif num == 5: # CRP
            return 6
        # Note: TrEC cells are absent in MLNs
        elif num == 6: # HEC
            return 8
        elif num == 4: # Vn
            return 9
        else:
            return -1


def reassigning_cluster_numbers(tissue):
    # Loading clustering results
    df = pd.read_csv(f"{tissue}/filtered_feature_bc_matrix/cluster_def_by_monocle_final.rev.csv")
    # Saving with new cluster numbers
    final = [convert_cluster_number(tissue, c) for c in df["cvalues"].tolist()]
    df_final = pd.DataFrame([df["cnames"].tolist(), final]).T
    df_final.columns = ["cnames", "cvalues"]
    df_final.to_csv(f"{tissue}/filtered_feature_bc_matrix/cluster_def_by_monocle_final_replaced.rev.csv", index=False)

In [126]:
reassigning_cluster_numbers('PLN')
reassigning_cluster_numbers('MLN')
reassigning_cluster_numbers('PP')

# 5. Exclude non-vascular endothelial cells

In [135]:
def filter_adata(
    adata,
    tissue,
    cluster_def_filename,
    cluster_nums,
    positive_barcodes=None,
    positibe_barcodes_tissue_num=None,
):
    # 1. Getting cells belonging to specified clusters
    # The cluster definition file path depends on tissues.
    if tissue in ['PLN', 'MLN', 'PP']:
        cluster_def_path = f"{tissue}/filtered_feature_bc_matrix/{cluster_def_filename}"
    else:
        cluster_def_path = cluster_def_filename # "cluster_def_by_monocle_k50.v2.csv"
    df_c = pd.read_csv(cluster_def_path)
    df_c_gs = [df_c[df_c["cvalues"]==n] for n in cluster_nums]
    df_c_g = pd.concat(df_c_gs)
    # The barcode format depends on tissues.
    if tissue in ['PLN', 'MLN', 'PP']:
        barcodes_g = df_c_g["cnames"].tolist()
    else:
        barcodes_g = [i[:-2] for i in df_c_g["cnames"].tolist()]

    # 2. Focusing on positive barcodes
    if positive_barcodes is not None:
        b = [i for i in positive_barcodes if i[-1]==str(positibe_barcodes_tissue_num)]
        barcodes_g = list(set(barcodes_g) & set([i[:-2] for i in b]))

    # 3. Filtering gex with 1&2
    barcodes = adata.obs.index.tolist()
    TF_g = [True if i in barcodes_g else False for i in barcodes]

    return adata[TF_g, :]

In [140]:
# Generate dataset only containing target cells
# PLN
print(f'PLN')
cluster_nums = [1, 2, 3, 4, 5, 6, 7, 8, 9]
adata_PLN_sub = filter_adata(
    adata_PLN,
    'PLN',
    'cluster_def_by_monocle_final_replaced.rev.csv',
    cluster_nums
)
print(adata_PLN.shape)
save_matrix(
    'PLN',
    adata_PLN_sub,
    out_barcodes_fn=f'barcodes_gex_{"-".join([str(n) for n in cluster_nums])}.rev.tsv',
    out_features_fn=None,
    out_matrix_fn=None
)

# MLN
print(f'MLN')
cluster_nums = [1, 2, 3, 4, 5, 6, 8, 9]
adata_MLN_sub = filter_adata(
    adata_MLN,
    'MLN',
    'cluster_def_by_monocle_final_replaced.rev.csv',
    cluster_nums
)
print(adata_MLN.shape)
save_matrix(
    'MLN',
    adata_MLN_sub,
    out_barcodes_fn=f'barcodes_gex_{"-".join([str(n) for n in cluster_nums])}.rev.tsv',
    out_features_fn=None,
    out_matrix_fn=None
)

# PP
print(f'PP')
cluster_nums = [1, 2, 3, 4, 5, 6, 8, 9]
adata_PP_sub = filter_adata(
    adata_PP,
    'PP',
    'cluster_def_by_monocle_final_replaced.rev.csv',
    cluster_nums
)
print(adata_PP.shape)
save_matrix(
    'PP',
    adata_PP_sub,
    out_barcodes_fn=f'barcodes_gex_{"-".join([str(n) for n in cluster_nums])}.rev.tsv',
    out_features_fn=None,
    out_matrix_fn=None
)

PLN
(7387, 20708)
MLN
(8703, 20313)
PP
(7985, 20568)


In [141]:
# Re-perform prefiltering
adata_PLN_re = prefiltering('PLN', 'barcodes_gex_1-2-3-4-5-6-7-8-9.rev.tsv')
save_matrix('PLN', adata_PLN_re)

adata_MLN_re = prefiltering('MLN', 'barcodes_gex_1-2-3-4-5-6-8-9.rev.tsv')
save_matrix('MLN', adata_MLN_re)

adata_PP_re = prefiltering('PP', 'barcodes_gex_1-2-3-4-5-6-8-9.rev.tsv')
save_matrix('PP', adata_PP_re)

... reading from cache file cache/PLN-filtered_feature_bc_matrix-matrix.h5ad
filtered out 566 cells that have less than 1000 genes expressed
filtered out 11608 genes that are detected in less than 3 cells
Shape of adata after prefiltering: (7089, 20677)
Shape of adata after excluding non-vascular endothelial cells: (6701, 20677)
... reading from cache file cache/MLN-filtered_feature_bc_matrix-matrix.h5ad
filtered out 799 cells that have less than 1000 genes expressed
filtered out 12007 genes that are detected in less than 3 cells
Shape of adata after prefiltering: (8377, 20278)
Shape of adata after excluding non-vascular endothelial cells: (7792, 20278)
... reading from cache file cache/PP-filtered_feature_bc_matrix-matrix.h5ad
filtered out 3599 cells that have less than 1000 genes expressed
filtered out 11902 genes that are detected in less than 3 cells
Shape of adata after prefiltering: (7674, 20383)
Shape of adata after excluding non-vascular endothelial cells: (7564, 20383)


# 6. Integration and visualization of three scRNA-seq data

## 6.1. Simple integration

In [5]:
rscript2 = f"""################################
# This is an R script.
# --------------------------------------------------------
# Filename: clustering_cells_combined.r
# --------------------------------------------------------
# How to run
# $ Rscript clustering_cells_combined.r
# --------------------------------------------------------
# outputs
# - {FigS8sub}.pdf
# - cluster_def_by_monocle_k50.v2.rev.csv
# - coordinate_by_monocle.v2.rev.csv
################################
# Library
library(monocle3)
library(ggplot2)
library(dplyr)

# Loading data
cds1 <- load_mm_data(
    mat_path = "PLN/filtered_feature_bc_matrix/matrix_gex.v2.rev.mtx",
    feature_anno_path = "PLN/filtered_feature_bc_matrix/features_gex.v2.rev.tsv",
    cell_anno_path = "PLN/filtered_feature_bc_matrix/barcodes_gex.v2.rev.tsv"
)
cds2 <- load_mm_data(
    mat_path = "MLN/filtered_feature_bc_matrix/matrix_gex.v2.rev.mtx",
    feature_anno_path = "MLN/filtered_feature_bc_matrix/features_gex.v2.rev.tsv",
    cell_anno_path = "MLN/filtered_feature_bc_matrix/barcodes_gex.v2.rev.tsv"
)
cds3 <- load_mm_data(
    mat_path = "PP/filtered_feature_bc_matrix/matrix_gex.v2.rev.mtx",
    feature_anno_path = "PP/filtered_feature_bc_matrix/features_gex.v2.rev.tsv",
    cell_anno_path = "PP/filtered_feature_bc_matrix/barcodes_gex.v2.rev.tsv"
)

# Combining preprocessing and remove batch effects
cds <- combine_cds(list(cds1, cds2, cds3))
names(rowData(cds)) <- "gene_short_name"
cds <- preprocess_cds(cds, num_dim = 20)
cds <- align_cds(cds, num_dim = 100, alignment_group = "sample")

# Dimension reduction
cds <- reduce_dimension(cds)

# Clustering and visualization
cds <- cluster_cells(cds, k=50)
pdf("{FigS4sub}.pdf")
print(plot_cells(cds))
dev.off()

# Saving clustering results
cnames <- names(clusters(cds))
cvalues <- as.numeric(as.character(clusters(cds)))
cmat <- cbind(cnames, cvalues)
write.csv(x=cmat, "cluster_def_by_monocle_k50.v2.rev.csv", row.names=F, quote=F)

# Saving coordinates
write.csv(x=reducedDims(cds)[["UMAP"]], "coordinate_by_monocle.v2.rev.csv", quote=F)
"""

f = open(f'clustering_cells_combined.r', 'w')
f.write(rscript2)
f.close()

In [ ]:
!Rscript clustering_cells_combined.r

## 6.2. Inspect the result

In [18]:
# Visualize current cluster definitions.
def plot_cluster(
    pos_path,
    cdef,
    filename=None,
    tissue="ALL",
    btype="Monocle",
    cxname="V1",
    cyname="V2",
    plot_width=800,
    plot_height=700,
    size=4
):
    # coordinates
    df = pd.read_csv(pos_path, index_col=0)
    x_ = df[cxname]
    y_ = df[cyname]

    # colors and legends
    # tissue idx
    tidx = "1" if tissue == "PLN" else "2" if tissue == "MLN" else "3" if tissue == "PP" else "4" if tissue == "ALL" else "-1"
    # definition
    color_map = {
        1:"#D0021B",
        3:"#BA98FF",
        4:"#FF789B",
        5:"#50E3C2",
        6:"#7ED321",
        7:"#4A90E2",
        8:"#F5A623",
        9:"#9013FE",
        -1: "black",
    }
    legend_map = {
        1:"Art", # Art
        3:"CapEC2'", # CapEC2'
        4:"CapEC2", # CapEC2
        5:"CapEC1", # CapEC1
        6:"CRP", # CRP
        7:"TrEC", # TrEC
        8:"HEC", # HEC
        9:"Vn", # Vn
        -1:"Others", # Others
    }
    # colors and legends
    x = []
    y = []
    color = []
    legend = []
    if btype == "Monocle":
        if tidx != "4":
            x = [x_[i] for i in df.index.tolist() if i+"_"+tidx in cdef.keys()]
            y = [y_[i] for i in df.index.tolist() if i+"_"+tidx in cdef.keys()]
            color = [color_map[cdef[i+"_"+tidx]] for i in df.index.tolist() if i+"_"+tidx in cdef.keys()]
            legend = [legend_map[cdef[i+"_"+tidx]] for i in df.index.tolist() if i+"_"+tidx in cdef.keys()]
        else:
            x = [x_[i] for i in df.index.tolist() if i in cdef.keys()]
            y = [y_[i] for i in df.index.tolist() if i in cdef.keys()]
            color = [color_map[cdef[i]] for i in df.index.tolist() if i in cdef.keys()]
            legend = [legend_map[cdef[i]] for i in df.index.tolist() if i in cdef.keys()]
    elif btype == "Signac":
        if tidx != "4":
            x = [x_[i] for i in df.index.tolist() if i.split("_")[1]+"_"+tidx in cdef.keys()]
            y = [y_[i] for i in df.index.tolist() if i.split("_")[1]+"_"+tidx in cdef.keys()]
            color = [color_map[cdef[i.split("_")[1]+"_"+tidx]] for i in df.index.tolist() if i.split("_")[1]+"_"+tidx in cdef.keys()]
            legend = [legend_map[cdef[i.split("_")[1]+"_"+tidx]] for i in df.index.tolist() if i.split("_")[1]+"_"+tidx in cdef.keys()]
        else:
            for i in df.index.tolist():
                k = i.split("_")[1]+"_1" if i.startswith("pln_") else i.split("_")[1]+"_2" if i.startswith("mln_") else i.split("_")[1]+"_3"
                if k in cdef.keys():
                    x.append(x_[i])
                    y.append(y_[i])
                    color.append(color_map[cdef[k]])
                    legend.append(legend_map[cdef[k]])

    source = ColumnDataSource(data=dict(x=x, y=y, color=color, legend=legend))

    p = figure(width=plot_width, height=plot_height, tools=["save"])
    p.circle('x', 'y', color='color', line_color=None, size=size, source=source, legend_group='legend')
    p.add_layout(p.legend[0], "right")
    p.xgrid.grid_line_alpha = 0
    p.ygrid.grid_line_alpha = 0

    # save
    p.background_fill_color = None
    p.border_fill_color = None
    export_png(p, filename=filename)

    # show
    output_notebook()
    show(p)

In [20]:
# 1. Loading cluster definition from cimbined data.
cdef = pd.read_csv("cluster_def_by_monocle_k50.v2.rev.csv", index_col=0)["cvalues"]

# 2. Loading cluster definition from each tissue
cdefsub1 = pd.read_csv("PLN/filtered_feature_bc_matrix/cluster_def_by_monocle_final_replaced.rev.csv", index_col=0)["cvalues"]
cdefsub1 = {k+"_1": cdefsub1[k] for k in cdefsub1.keys()}
cdefsub2 = pd.read_csv("MLN/filtered_feature_bc_matrix/cluster_def_by_monocle_final_replaced.rev.csv", index_col=0)["cvalues"]
cdefsub2 = {k+"_2": cdefsub2[k] for k in cdefsub2.keys()}
cdefsub3 = pd.read_csv("PP/filtered_feature_bc_matrix/cluster_def_by_monocle_final_replaced.rev.csv", index_col=0)["cvalues"]
cdefsub3 = {k+"_3": cdefsub3[k] for k in cdefsub3.keys()}

# 3. Mapping definition
for k in cdef.keys():
    n = cdefsub1[k] if k in cdefsub1.keys() else cdefsub2[k] if k in cdefsub2 else cdefsub3[k]
    # Merge CapEC2 into a single cluster.
    if n in [2, 3, 4]:
        cdef[k] = 4
    else:
        cdef[k] = n

# 4. Automatic annotation based on marginal distributions
# 4-1: Function to obtain cells within a square region centered on `pos_self`.
def get_rect(pos, val, pos_self):
    pos = pos[pos["V1"] < pos_self[0] + val]
    pos = pos[pos["V1"] > pos_self[0] - val]
    pos = pos[pos["V2"] < pos_self[1] + val]
    pos = pos[pos["V2"] > pos_self[1] - val]
    return pos
# 4-2: Loading coordinates derived from the integrated data.
pos = pd.read_csv("coordinate_by_monocle.v2.rev.csv", index_col=0)
# 4-3: Assign known clusters or other clusters for the rest of cells
for c in list(cdef.keys()):
    # Coordinates of target cell.
    pos_self = pos.loc[c, :].tolist() # Coordinates of target cell
    # Retrieve cells within the smallest square region that contains at least one cell.
    pos_ = pos
    for i in [0.5, 1, 2, 3, 4, 5, 6, 7, 8, 9]:
        _ = get_rect(pos_, i, pos_self)
        if len(_) > 0:
            break
    pos_ = _
    # Assign -1 as the cluster number if the square region described above does not exist.
    if len(_) == 0:
        cdef[c] = -1
        continue
    # Assign the most frequent cluster number among the obtained cell population.
    ns = [cdef[i] for i in pos_.index.tolist()]
    vals = sorted(list(set(ns)))
    nsc = [ns.count(i) for i in vals]
    cdef[c] = vals[np.argmax(nsc)]
    if 0.5 <= pos_self[0] and cdef[c] == 5:
        cdef[c] = 4

In [21]:
# Supplementary_Fig4(a).png
plot_cluster(
    "coordinate_by_monocle.v2.rev.csv",
    cdef,
    f'{FigS8a}.png',
    tissue="ALL",
    btype="Monocle",
    cxname="V1",
    cyname="V2",
    plot_width=800,
    plot_height=700,
    size=4
)

Loading BokehJS ...

## 6.3. Subclustering the CapEC2

In [165]:
# Extracting CapEC2 cells
CapEC2s = [k[:-2] for k in cdef.keys() if cdef[k]==4]

# PLN
barcodes = adata_PLN_re.obs.index.tolist()
TF_g = [True if i in CapEC2s else False for i in barcodes]
adata_PLN_re_sub = adata_PLN_re[TF_g, :]
save_matrix(
    'PLN',
    adata_PLN_re_sub,
    out_barcodes_fn='barcodes_gex.v2.subset.rev.tsv',
    out_features_fn='features_gex.v2.subset.rev.tsv',
    out_matrix_fn='matrix_gex.v2.subset.rev.mtx'
)

# MLN
barcodes = adata_MLN_re.obs.index.tolist()
TF_g = [True if i in CapEC2s else False for i in barcodes]
adata_MLN_re_sub = adata_MLN_re[TF_g, :]
save_matrix(
    'MLN',
    adata_MLN_re_sub,
    out_barcodes_fn='barcodes_gex.v2.subset.rev.tsv',
    out_features_fn='features_gex.v2.subset.rev.tsv',
    out_matrix_fn='matrix_gex.v2.subset.rev.mtx'
)

# PP
barcodes = adata_PP_re.obs.index.tolist()
TF_g = [True if i in CapEC2s else False for i in barcodes]
adata_PP_re_sub = adata_PP_re[TF_g, :]
save_matrix(
    'PP',
    adata_PP_re_sub,
    out_barcodes_fn='barcodes_gex.v2.subset.rev.tsv',
    out_features_fn='features_gex.v2.subset.rev.tsv',
    out_matrix_fn='matrix_gex.v2.subset.rev.mtx'
)

In [29]:
rscript3 = f"""################################
# This is an R script.
# --------------------------------------------------------
# Filename: subclustering_cells.r
# --------------------------------------------------------
# How to run
# $ Rscript subclustering_cells.r
# --------------------------------------------------------
# outputs
# - {FigS8b}.pdf
# - {FigS9}.pdf
# - cluster_def_by_monocle_k30_sub.rev.csv
# - coordinate_by_monocle.v2_sub.rev.csv
################################
# Library
library(monocle3)
library(ggplot2)
library(dplyr)

# Loading data
cds1 <- load_mm_data(
    mat_path = "PLN/filtered_feature_bc_matrix/matrix_gex.v2.subset.rev.mtx",
    feature_anno_path = "PLN/filtered_feature_bc_matrix/features_gex.v2.subset.rev.tsv",
    cell_anno_path = "PLN/filtered_feature_bc_matrix/barcodes_gex.v2.subset.rev.tsv"
)

cds2 <- load_mm_data(
    mat_path = "MLN/filtered_feature_bc_matrix/matrix_gex.v2.subset.rev.mtx",
    feature_anno_path = "MLN/filtered_feature_bc_matrix/features_gex.v2.subset.rev.tsv",
    cell_anno_path = "MLN/filtered_feature_bc_matrix/barcodes_gex.v2.subset.rev.tsv"
)

cds3 <- load_mm_data(
    mat_path = "PP/filtered_feature_bc_matrix/matrix_gex.v2.subset.rev.mtx",
    feature_anno_path = "PP/filtered_feature_bc_matrix/features_gex.v2.subset.rev.tsv",
    cell_anno_path = "PP/filtered_feature_bc_matrix/barcodes_gex.v2.subset.rev.tsv"
)

# Combining preprocessing and remove batch effects
cds <- combine_cds(list(cds1, cds2, cds3))
names(rowData(cds)) <- "gene_short_name"
cds <- preprocess_cds(cds, num_dim = 20)
cds <- align_cds(cds, num_dim = 100, alignment_group = "sample")

# Dimension reduction
cds <- reduce_dimension(cds)

# Checking marker gene expression (CapEC2)
pdf("{FigS9}.pdf")
print(plot_cells(cds, genes=c("Atf3", "Cxcl1", "Egr1", "Fos", "Fosb", "Jun", "Junb", "Jund", "Nfkbia", "Nr4a1", "Sgk1")))
dev.off()

# Clustering and visualization
cds <- cluster_cells(cds, k=30)
pdf("{FigS8b}.pdf")
print(plot_cells(cds))
dev.off()

# Saving clustering results
cnames <- names(clusters(cds))
cvalues <- as.numeric(as.character(clusters(cds)))
cmat <- cbind(cnames, cvalues)
write.csv(x=cmat, "cluster_def_by_monocle_k30_sub.rev.csv", row.names=F, quote=F)

# Saving coordinates
write.csv(x=reducedDims(cds)[["UMAP"]], "coordinate_by_monocle.v2_sub.rev.csv", quote=F)
"""

f = open(f'subclustering_cells.r', 'w')
f.write(rscript3)
f.close()

In [ ]:
!Rscript subclustering_cells.r

## 6.4. Visualize combined scRNA-Seq data

In [22]:
# 1. Loading cluster definition from cimbined data.
cdef_CapEC2 = pd.read_csv("cluster_def_by_monocle_k30_sub.rev.csv", index_col=0)["cvalues"]

# 2. Update the original definitions by applying the subclustering results.
for c in list(cdef.keys()):
    if c in cdef_CapEC2.keys():
        # CapEC2
        if cdef_CapEC2[c] == 1:
            continue
        # CapEC2'
        else:
            cdef[c] = 3

# 3. Automatic annotation based on marginal distributions
# 3-1: Loading coordinates derived from the integrated data.
pos = pd.read_csv("coordinate_by_monocle.v2.rev.csv", index_col=0)
# 3-2: Assign known clusters or other clusters for the rest of cells
for c in list(cdef.keys()):
    # Coordinates of target cell.
    pos_self = pos.loc[c, :].tolist() # Coordinates of target cell
    # Retrieve cells within the smallest square region that contains at least one cell.
    pos_ = pos
    for i in [0.5, 1, 2, 3, 4, 5, 6, 7, 8, 9]:
        _ = get_rect(pos_, i, pos_self)
        if len(_) > 0:
            break
    pos_ = _
    # Assign -1 as the cluster number if the square region described above does not exist.
    if len(_) == 0:
        cdef[c] = -1
        continue
    # Assign the most frequent cluster number among the obtained cell population.
    ns = [cdef[i] for i in pos_.index.tolist()]
    vals = sorted(list(set(ns)))
    nsc = [ns.count(i) for i in vals]
    cdef[c] = vals[np.argmax(nsc)]
    # if 0.5 <= pos_self[0] and cdef[c] == 5:
    #     cdef[c] = 4

# 4. Fig1(b)_GEX_All.png
plot_cluster(
    "coordinate_by_monocle.v2.rev.csv",
    cdef,
    f'{Fig1b_GEX_ALL}.png',
    tissue="ALL",
    btype="Monocle",
    cxname="V1",
    cyname="V2",
    plot_width=800,
    plot_height=700,
    size=4
)

# 5. Save the cluster definitions.
# pickle
cdef = dict(cdef)
with open("cluster_def_by_monocle_final.v2.1.rev.pkl", "wb") as tf:
    pickle.dump(cdef, tf)
# csv
df_cdef = df = pd.DataFrame.from_dict(cdef, orient='index')
df_cdef.columns = ['monocle_clusters']
df_cdef.to_csv(f'cluster_def_by_monocle_final.v2.1.rev.csv')

Loading BokehJS ...

## 6.5. Visualize each scRNA-Seq data

In [ ]:
rscript1 = f"""################################
# This is an R script.
# --------------------------------------------------------
# Filename: visualizing_clusters_gex.r
# --------------------------------------------------------
# How to run
# $ Rscript visualizing_clusters_gex.r
# --------------------------------------------------------
# outputs
# - PLN/filtered_feature_bc_matrix/coordinate_by_monocle.v2.rev.csv
# - MLN/filtered_feature_bc_matrix/coordinate_by_monocle.v2.rev.csv
# - PP/filtered_feature_bc_matrix/coordinate_by_monocle.v2.rev.csv
################################
# Library
library(monocle3)
library(ggplot2)
library(dplyr)

# Loading data
cds1 <- load_mm_data(
    mat_path = "PLN/filtered_feature_bc_matrix/matrix_gex.v2.rev.mtx",
    feature_anno_path = "PLN/filtered_feature_bc_matrix/features_gex.v2.rev.tsv", 
    cell_anno_path = "PLN/filtered_feature_bc_matrix/barcodes_gex.v2.rev.tsv"
)
cds2 <- load_mm_data(
    mat_path = "MLN/filtered_feature_bc_matrix/matrix_gex.v2.rev.mtx",
    feature_anno_path = "MLN/filtered_feature_bc_matrix/features_gex.v2.rev.tsv", 
    cell_anno_path = "MLN/filtered_feature_bc_matrix/barcodes_gex.v2.rev.tsv"
)
cds3 <- load_mm_data(
    mat_path = "PP/filtered_feature_bc_matrix/matrix_gex.v2.rev.mtx", 
    feature_anno_path = "PP/filtered_feature_bc_matrix/features_gex.v2.rev.tsv", 
    cell_anno_path = "PP/filtered_feature_bc_matrix/barcodes_gex.v2.rev.tsv"
)

# Saving coordinates
cds1 <- preprocess_cds(cds1, num_dim = 20)
cds1 <- reduce_dimension(cds1)
write.csv(x=reducedDims(cds1)[["UMAP"]], "PLN/filtered_feature_bc_matrix/coordinate_by_monocle.v2.rev.csv", quote=F)

cds2 <- preprocess_cds(cds2, num_dim = 20)
cds2 <- reduce_dimension(cds2)
write.csv(x=reducedDims(cds2)[["UMAP"]], "MLN/filtered_feature_bc_matrix/coordinate_by_monocle.v2.rev.csv", quote=F)

cds3 <- preprocess_cds(cds3, num_dim = 20)
cds3 <- reduce_dimension(cds3, reduction_method="tSNE")
write.csv(x=reducedDims(cds3)[["UMAP"]], "PP/filtered_feature_bc_matrix/coordinate_by_monocle.v2.rev.csv", quote=F)
"""

f = open(f'visualizing_clusters_gex.r', 'w')
f.write(rscript1)
f.close()

In [ ]:
!Rscript visualizing_clusters_gex.r

In [ ]:
# Fig1(b)_GEX_PLN.png
plot_cluster(
    "PLN/filtered_feature_bc_matrix/coordinate_by_monocle.v2.rev.csv",
    cdef,
    f'{Fig1b_GEX_PLN}.png',
    tissue="PLN",
    btype="Monocle",
    cxname="V1",
    cyname="V2",
    plot_width=850,
    plot_height=400,
    size=4
)

In [ ]:
# Fig1(b)_GEX_MLN.png
plot_cluster(
    "MLN/filtered_feature_bc_matrix/coordinate_by_monocle.v2.rev.csv",
    cdef,
    f'{Fig1b_GEX_MLN}.png',
    tissue="MLN",
    btype="Monocle",
    cxname="V1",
    cyname="V2",
    plot_width=900,
    plot_height=600,
    size=4
)

In [ ]:
# Fig1(b)_GEX_PP.png
plot_cluster(
    "PP/filtered_feature_bc_matrix/coordinate_by_monocle.v2.rev.csv",
    cdef,
    f'{Fig1b_GEX_PP}.png',
    tissue="PP",
    btype="Monocle",
    cxname="V1",
    cyname="V2",
    plot_width=850,
    plot_height=700,
    size=4
)

# 7. Integration and visualization of three scATAC-seq data

In [1]:
# copy the source data
shellscript1 = f"""################################
# This is a shellscript.
# --------------------------------------------------------
# Filename: copy_matrix_data.sh
# --------------------------------------------------------
# How to run
# $ bash copy_matrix_data.sh
################################
mkdir -p Seurat/PLN/data2
cp PLN/filtered_feature_bc_matrix/barcodes_gex.v2.rev.tsv Seurat/PLN/data2
cp PLN/filtered_feature_bc_matrix/features_gex.v2.rev.tsv Seurat/PLN/data2
cp PLN/filtered_feature_bc_matrix/matrix_gex.v2.rev.mtx Seurat/PLN/data2
cd Seurat/PLN/data2
gzip barcodes_gex.v2.rev.tsv features_gex.v2.rev.tsv matrix_gex.rev.v2.mtx
mv barcodes_gex.v2.rev.tsv.gz barcodes.tsv.gz
mv features_gex.v2.rev.tsv.gz features.tsv.gz
mv matrix_gex.v2.rev.mtx.gz matrix.mtx.gz
cd ../../../

mkdir -p Seurat/MLN/data2
cp MLN/filtered_feature_bc_matrix/barcodes_gex.v2.rev.tsv Seurat/MLN/data2
cp MLN/filtered_feature_bc_matrix/features_gex.v2.rev.tsv Seurat/MLN/data2
cp MLN/filtered_feature_bc_matrix/matrix_gex.v2.rev.mtx Seurat/MLN/data2
cd Seurat/MLN/data2
gzip barcodes_gex.v2.rev.tsv features_gex.v2.rev.tsv matrix_gex.rev.v2.mtx
mv barcodes_gex.v2.rev.tsv.gz barcodes.tsv.gz
mv features_gex.v2.rev.tsv.gz features.tsv.gz
mv matrix_gex.v2.rev.mtx.gz matrix.mtx.gz
cd ../../../

mkdir -p Seurat/PP/data3
cp PP/filtered_feature_bc_matrix/barcodes_gex.v2.rev.tsv Seurat/PP/data2
cp PP/filtered_feature_bc_matrix/features_gex.v2.rev.tsv Seurat/PP/data2
cp PP/filtered_feature_bc_matrix/matrix_gex.v2.rev.mtx Seurat/PP/data2
cd Seurat/PP/data2
gzip barcodes_gex.v2.rev.tsv features_gex.v2.rev.tsv matrix_gex.rev.v2.mtx
mv barcodes_gex.v2.rev.tsv.gz barcodes.tsv.gz
mv features_gex.v2.rev.tsv.gz features.tsv.gz
mv matrix_gex.v2.rev.mtx.gz matrix.mtx.gz
cd ../../../
"""

f = open(f'copy_matrix_data.sh', 'w')
f.write(shellscript1)
f.close()

In [ ]:
!bash copy_matrix_data.sh

In [ ]:
# Calculate coordinates based on ATAC data.
rscript2 = f"""################################
# This is an R script.
# --------------------------------------------------------
# Filename: visualizing_clusters_atac.r
# --------------------------------------------------------
# How to run
# $ Rscript visualizing_clusters_atac.r
# --------------------------------------------------------
# inputs
# - Signac/PLN/atac_peaks.bed
# - Signac/PLN/atac_fragments.tsv.gz
# - Signac/MLN/atac_peaks.bed
# - Signac/MLN/atac_fragments.tsv.gz
# - Signac/PP/atac_peaks.bed
# - Signac/PP/atac_fragments.tsv.gz
# - Seurat/PLN/data2/*
# - Seurat/MLN/data2/*
# - Seurat/PP/data3/*
# --------------------------------------------------------
# outputs
# - Signac/PLN/pln.atac.v2.RData
# - Signac/PLN/coordinate(atac)_by_Signac.v2.csv
# - Signac/MLN/mln.atac.v2.RData
# - Signac/MLN/coordinate(atac)_by_Signac.v2.csv
# - Signac/PP/pp.atac.v2.RData
# - Signac/PP/coordinate(atac)_by_Signac.v2.csv
# - Signac/atac.v2.RData
# - Signac/coordinate(atac)_by_Signac.v2.csv
################################
# 1-1. Library
library(Signac)
library(Seurat)
library(GenomicRanges)
library(future)
library(GenomeInfoDb)
library(EnsDb.Mmusculus.v79)
library(ggplot2)
library(patchwork)
set.seed(1234)

# 1-2. Memory
options(future.globals.maxSize = 16000 * 1024^2)

# 2-1. Read in peak sets
peaks.pln <- read.table(
  file = "Signac/PLN/atac_peaks.bed",
  col.names = c("chr", "start", "end")
)
peaks.mln <- read.table(
  file = "Signac/MLN/atac_peaks.bed",
  col.names = c("chr", "start", "end")
)
peaks.pp <- read.table(
  file = "Signac/PP/atac_peaks.bed",
  col.names = c("chr", "start", "end")
)

# 2-2. Convert to genomic ranges
gr.pln <- makeGRangesFromDataFrame(peaks.pln)
gr.mln <- makeGRangesFromDataFrame(peaks.mln)
gr.pp <- makeGRangesFromDataFrame(peaks.pp)

# 2-3. Create a unified set of peaks to quantify in each dataset
combined.peaks <- reduce(x = c(gr.pln, gr.mln, gr.pp))

# 2-4. Filter out bad peaks based on length
peakwidths <- width(combined.peaks)
combined.peaks <- combined.peaks[peakwidths  < 10000 & peakwidths > 20]

# 3-1. Loading common cells
counts.pln <- Read10X("Seurat/PLN/data2/")
counts.mln <- Read10X("Seurat/MLN/data2/")
counts.pp <- Read10X("Seurat/PP/data3/")

# 3-2. Create fragment objects
frags.pln <- CreateFragmentObject(
  path = "Signac/PLN/atac_fragments.tsv.gz",
  cells = colnames(counts.pln)
)
frags.mln <- CreateFragmentObject(
  path = "Signac/MLN/atac_fragments.tsv.gz",
  cells = colnames(counts.mln)
)
frags.pp <- CreateFragmentObject(
  path = "Signac/PP/atac_fragments.tsv.gz",
  cells = colnames(counts.pp)
)

# 3-3. Create count objects
pln.counts <- FeatureMatrix(
  fragments = frags.pln,
  features = combined.peaks,
  cells = colnames(counts.pln)
)
mln.counts <- FeatureMatrix(
  fragments = frags.mln,
  features = combined.peaks,
  cells = colnames(counts.mln)
)
pp.counts <- FeatureMatrix(
  fragments = frags.pp,
  features = combined.peaks,
  cells = colnames(counts.pp)
)

# 3-4. Create Seurat objects
pln_assay <- CreateChromatinAssay(
    pln.counts,
    sep = c(":", "-"),
    genome = "mm10",
    fragments = frags.pln
)
pln <- CreateSeuratObject(
    pln_assay,
    assay = 'peaks',
    project = 'ATAC'
)
mln_assay <- CreateChromatinAssay(
    mln.counts,
    sep = c(":", "-"),
    genome = "mm10",
    fragments = frags.mln
)
mln <- CreateSeuratObject(
    mln_assay,
    assay = 'peaks',
    project = 'ATAC'
)
pp_assay <- CreateChromatinAssay(
    pp.counts,
    sep = c(":", "-"),
    genome = "mm10",
    fragments = frags.pp
)
pp <- CreateSeuratObject(
    pp_assay,
    assay = 'peaks',
    project = 'ATAC'
)

# 4-1. Annotation
annotations <- GetGRangesFromEnsDb(ensdb = EnsDb.Mmusculus.v79)
seqlevelsStyle(annotations) <- 'UCSC'
Annotation(pln) <- annotations
Annotation(mln) <- annotations
Annotation(pp) <- annotations

# 4-2. Prefiltering (1)
pln <- NucleosomeSignal(object = pln)
pln <- TSSEnrichment(pln, fast = FALSE)
pln <- subset(
  x = pln,
  subset = nucleosome_signal < 4 &
    TSS.enrichment > 2
)
mln <- NucleosomeSignal(object = mln)
mln <- TSSEnrichment(mln, fast = FALSE)
mln <- subset(
  x = mln,
  subset = nucleosome_signal < 4 &
    TSS.enrichment > 2
)
pp <- NucleosomeSignal(object = pp)
pp <- TSSEnrichment(pp, fast = FALSE)
pp <- subset(
  x = pp,
  subset = nucleosome_signal < 4 &
    TSS.enrichment > 2
)

# 4-3. Prefiltering (2)
pln <- RunTFIDF(pln)
pln <- FindTopFeatures(pln, min.cutoff = 'q0')
pln <- RunSVD(object = pln)
pln <- RunUMAP(
  object = pln,
  reduction = 'lsi',
  dims = 2:30
)
mln <- RunTFIDF(mln)
mln <- FindTopFeatures(mln, min.cutoff = 'q0')
mln <- RunSVD(object = mln)
mln <- RunUMAP(
  object = mln,
  reduction = 'lsi',
  dims = 2:30
)
pp <- RunTFIDF(pp)
pp <- FindTopFeatures(pp, min.cutoff = 'q0')
pp <- RunSVD(object = pp)
pp <- RunUMAP(
  object = pp,
  reduction = 'lsi',
  dims = 2:30
)

# 5. Save and export
# 5-1. Save objects
save(pln, file="Signac/PLN/pln.atac.v2.RData")
save(mln, file="Signac/MLN/mln.atac.v2.RData")
save(pp, file="Signac/PP/pp.atac.v2.RData")

# 5-2. Export coordinates
write.csv(x=pln[["umap"]][[]], "Signac/PLN/coordinate(atac)_by_Signac.v2.csv", quote=F)
write.csv(x=mln[["umap"]][[]], "Signac/MLN/coordinate(atac)_by_Signac.v2.csv", quote=F)
write.csv(x=pp[["umap"]][[]], "Signac/PP/coordinate(atac)_by_Signac.v2.csv", quote=F)

# 6. Integrate three tissues
# 6-1. Add information to identify dataset of origin
pln$dataset <- 'pln'
mln$dataset <- 'mln'
pp$dataset <- 'pp'

# 6-2. Merge all datasets, adding a cell ID to make sure cell names are unique
combined <- merge(
  x = pln,
  y = list(mln, pp),
  add.cell.ids = c("pln", "mln", "pp")
)

# 6-3. Preprocessing
combined <- RunTFIDF(combined)
combined <- FindTopFeatures(combined, min.cutoff = 20)
combined <- RunSVD(combined)
combined <- RunUMAP(combined, dims = 2:50, reduction = 'lsi')

# 6-4. Save and export
save(combined, file="Signac/atac.v2.RData")
write.csv(x=combined[["umap"]][[]], "Signac/coordinate(atac)_by_Signac.v2.csv", quote=F)
"""

f = open(f'visualizing_clusters_atac.r', 'w')
f.write(rscript2)
f.close()

In [ ]:
!Rscript visualizing_clusters_atac.r

In [ ]:
# Visualization
# Fig1(b)_ATAC_PLN.png
plot_cluster(
    "Signac/PLN/coordinate(atac)_by_Signac.v2.csv",
    cdef,
    f'{Fig1b_ATAC_PLN}.png',
    tissue="PLN",
    btype="Monocle",
    cxname="UMAP_1",
    cyname="UMAP_2",
    plot_width=700,
    plot_height=700,
    size=4
)

In [ ]:
# Fig1(b)_ATAC_MLN.png
plot_cluster(
    "Signac/MLN/coordinate(atac)_by_Signac.v2.csv",
    cdef,
    f'{Fig1b_ATAC_MLN}.png',
    tissue="MLN",
    btype="Monocle",
    cxname="UMAP_1",
    cyname="UMAP_2",
    plot_width=700,
    plot_height=700,
    size=4
)

In [ ]:
# Fig1(b)_ATAC_PP.png
plot_cluster(
    "Signac/PP/coordinate(atac)_by_Signac.v2.csv",
    cdef,
    f'{Fig1b_ATAC_PP}.png',
    tissue="PP",
    btype="Monocle",
    cxname="UMAP_1",
    cyname="UMAP_2",
    plot_width=700,
    plot_height=700,
    size=4
)

In [ ]:
# Fig1(b)_ATAC_ALL.png
plot_cluster(
    "Signac/coordinate(atac)_by_Signac.v2.csv",
    cdef,
    f'{Fig1b_ATAC_ALL}.png',
    tissue="ALL",
    btype="Signac",
    cxname="UMAP_1",
    cyname="UMAP_2",
    plot_width=800,
    plot_height=700,
    size=4
)

# 8. Visualize clusters based on both data.

In [ ]:
# Calculate coordinates based on both data.
rscript3 = f"""################################
# This is an R script.
# --------------------------------------------------------
# Filename: visualizing_clusters_wnn.r
# --------------------------------------------------------
# How to run
# $ Rscript visualizing_clusters_wnn.r
# --------------------------------------------------------
# inputs
# - PLN/filtered_feature_bc_matrix/matrix_gex.v2.rev.mtx
# - PLN/filtered_feature_bc_matrix/features_gex.v2.rev.tsv
# - PLN/filtered_feature_bc_matrix/barcodes_gex.v2.rev.tsv
# - MLN/filtered_feature_bc_matrix/matrix_gex.v2.rev.mtx
# - MLN/filtered_feature_bc_matrix/features_gex.v2.rev.tsv
# - MLN/filtered_feature_bc_matrix/barcodes_gex.v2.rev.tsv
# - PP/filtered_feature_bc_matrix/matrix_gex.v2.rev.mtx
# - PP/filtered_feature_bc_matrix/features_gex.v2.rev.tsv
# - PP/filtered_feature_bc_matrix/barcodes_gex.v2.rev.tsv
# - Signac/PLN/pln.atac.v2.RData
# - Signac/MLN/mln.atac.v2.RData
# - Signac/PP/pp.atac.v2.RData
# - Signac/atac.v2.RData
# --------------------------------------------------------
# outputs
# - Seurat/PLN/out/gex.atac.v2.RData
# - Seurat/MLN/out/gex.atac.v2.RData
# - Seurat/PP/out/gex.atac.v2.RData
# - Seurat/out/gex.atac.v2.RData
# - Seurat/PLN/out/coordinate(wnn)_by_Seurat.v2.csv
# - Seurat/MLN/out/coordinate(wnn)_by_Seurat.v2.csv
# - Seurat/PP/out/coordinate(wnn)_by_Seurat.v2.csv
# - Seurat/out/coordinate(wnn)_by_Seurat.v2.csv
################################
# 1-1. Library
library(monocle3)
library(ggplot2)
library(dplyr)
library(Signac)
library(Seurat)
library(GenomicRanges)
library(future)
library(GenomeInfoDb)
library(EnsDb.Mmusculus.v79)
library(patchwork)
set.seed(1234)
library(SeuratWrappers) # Object conversion
library(stringr) # text handling

# 1-2. memory
options(future.globals.maxSize = 16000 * 1024^2)

# 2. Loading Seurat objects
load("Signac/PLN/pln.atac.v2.RData") # pln
load("Signac/MLN/mln.atac.v2.RData") # mln
load("Signac/PP/pp.atac.v2.RData") # pp
load("Signac/atac.v2.RData") # combined

# 3. Preprocessing GEX data
# 3-1. Load gex data from each tissue
cds1 <- load_mm_data(
    mat_path = "PLN/filtered_feature_bc_matrix/matrix_gex.v2.rev.mtx", 
    feature_anno_path = "PLN/filtered_feature_bc_matrix/features_gex.v2.rev.tsv", 
    cell_anno_path = "PLN/filtered_feature_bc_matrix/barcodes_gex.v2.rev.tsv"
)
cds2 <- load_mm_data(
    mat_path = "MLN/filtered_feature_bc_matrix/matrix_gex.v2.rev.mtx", 
    feature_anno_path = "MLN/filtered_feature_bc_matrix/features_gex.v2.rev.tsv", 
    cell_anno_path = "MLN/filtered_feature_bc_matrix/barcodes_gex.v2.rev.tsv"
)
cds3 <- load_mm_data(
    mat_path = "PP/filtered_feature_bc_matrix/matrix_gex.v2.rev.mtx", 
    feature_anno_path = "PP/filtered_feature_bc_matrix/features_gex.v2.rev.tsv", 
    cell_anno_path = "PP/filtered_feature_bc_matrix/barcodes_gex.v2.rev.tsv"
)

# 3-2. Rename
colnames(cds1) <- str_c("pln_", colnames(cds1), sep="")
colnames(cds2) <- str_c("mln_", colnames(cds2), sep="")
colnames(cds3) <- str_c("pp_", colnames(cds3), sep="")

# 3-3. Cell selection
cds1 <- cds1[, colnames(cds1) %in% rownames(combined@meta.data)]
cds2 <- cds2[, colnames(cds2) %in% rownames(combined@meta.data)]
cds3 <- cds3[, colnames(cds3) %in% rownames(combined@meta.data)]

# 3-4. Remove batch effect
cds <- combine_cds(list(cds1, cds2, cds3))
cds <- preprocess_cds(cds, num_dim = 20) # =PCA
cds <- align_cds(cds, num_dim = 100, alignment_group = "sample") # =Correct PCA

# 3-5. Rename
colnames(cds) <- str_replace(colnames(cds), pattern="_1", replacement="")
colnames(cds) <- str_replace(colnames(cds), pattern="_2", replacement="")
colnames(cds) <- str_replace(colnames(cds), pattern="_3", replacement="")

# 3-6. Rename
colnames(cds1) <- str_replace(colnames(cds1), pattern="pln_", replacement="")
colnames(cds2) <- str_replace(colnames(cds2), pattern="mln_", replacement="")
colnames(cds3) <- str_replace(colnames(cds3), pattern="pp_", replacement="")

# 3-7. Convert to Seurat object
cds1.seurat <- as.Seurat(cds1, assay = NULL)
cds2.seurat <- as.Seurat(cds2, assay = NULL)
cds3.seurat <- as.Seurat(cds3, assay = NULL)
cds.seurat <- as.Seurat(cds, assay = NULL)

# 4. WNN (each tissue)
# 4-1. Add ATAC
cds1.seurat[["ATAC"]] <- pln[["peaks"]]
cds2.seurat[["ATAC"]] <- mln[["peaks"]]
cds3.seurat[["ATAC"]] <- pp[["peaks"]]

# 4-2. RNA analysis
cds1.seurat <- SCTransform(cds1.seurat, verbose = FALSE, assay="originalexp") %>% RunPCA() %>% RunUMAP(dims = 1:50, reduction.name = 'umap.rna', reduction.key = 'rnaUMAP_')
cds2.seurat <- SCTransform(cds2.seurat, verbose = FALSE, assay="originalexp") %>% RunPCA() %>% RunUMAP(dims = 1:50, reduction.name = 'umap.rna', reduction.key = 'rnaUMAP_')
cds3.seurat <- SCTransform(cds3.seurat, verbose = FALSE, assay="originalexp") %>% RunPCA() %>% RunUMAP(dims = 1:50, reduction.name = 'umap.rna', reduction.key = 'rnaUMAP_')

# 4-3. ATAC analysis
DefaultAssay(cds1.seurat) <- "ATAC"
cds1.seurat <- RunTFIDF(cds1.seurat)
cds1.seurat <- FindTopFeatures(cds1.seurat, min.cutoff = 'q0')
cds1.seurat <- RunSVD(cds1.seurat)
cds1.seurat <- RunUMAP(cds1.seurat, reduction = 'lsi', dims = 2:50, reduction.name = "umap.atac", reduction.key = "atacUMAP_")
DefaultAssay(cds2.seurat) <- "ATAC"
cds2.seurat <- RunTFIDF(cds2.seurat)
cds2.seurat <- FindTopFeatures(cds2.seurat, min.cutoff = 'q0')
cds2.seurat <- RunSVD(cds2.seurat)
cds2.seurat <- RunUMAP(cds2.seurat, reduction = 'lsi', dims = 2:50, reduction.name = "umap.atac", reduction.key = "atacUMAP_")
DefaultAssay(cds3.seurat) <- "ATAC"
cds3.seurat <- RunTFIDF(cds3.seurat)
cds3.seurat <- FindTopFeatures(cds3.seurat, min.cutoff = 'q0')
cds3.seurat <- RunSVD(cds3.seurat)
cds3.seurat <- RunUMAP(cds3.seurat, reduction = 'lsi', dims = 2:50, reduction.name = "umap.atac", reduction.key = "atacUMAP_")

# 4-4. WNN
cds1.seurat <- FindMultiModalNeighbors(cds1.seurat, reduction.list = list("pca", "lsi"), dims.list = list(1:50, 2:50))
cds1.seurat <- RunUMAP(cds1.seurat, nn.name = "weighted.nn", reduction.name = "wnn.umap", reduction.key = "wnnUMAP_")
cds2.seurat <- FindMultiModalNeighbors(cds2.seurat, reduction.list = list("pca", "lsi"), dims.list = list(1:50, 2:50))
cds2.seurat <- RunUMAP(cds2.seurat, nn.name = "weighted.nn", reduction.name = "wnn.umap", reduction.key = "wnnUMAP_")
cds3.seurat <- FindMultiModalNeighbors(cds3.seurat, reduction.list = list("pca", "lsi"), dims.list = list(1:50, 2:50))
cds3.seurat <- RunUMAP(cds3.seurat, nn.name = "weighted.nn", reduction.name = "wnn.umap", reduction.key = "wnnUMAP_")

# 4-5. Save
save(cds1.seurat, file="Seurat/PLN/out/gex.atac.v2.RData")
save(cds2.seurat, file="Seurat/MLN/out/gex.atac.v2.RData")
save(cds3.seurat, file="Seurat/PP/out/gex.atac.v2.RData")

# 4-6. Export coordinate
write.csv(x=cds1.seurat[["wnn.umap"]][[]], "Seurat/PLN/out/coordinate(wnn)_by_Seurat.v2.csv", quote=F)
write.csv(x=cds2.seurat[["wnn.umap"]][[]], "Seurat/MLN/out/coordinate(wnn)_by_Seurat.v2.csv", quote=F)
write.csv(x=cds3.seurat[["wnn.umap"]][[]], "Seurat/PP/out/coordinate(wnn)_by_Seurat.v2.csv", quote=F)

# 5. WNN (Integrated data)
# 5-1. Add new attribute
cds.seurat[["ATAC"]] <- combined[["peaks"]]

# 5-2. Reduce dimension (GEX)
cds.seurat <- RunUMAP(object=cds.seurat, reduction='Aligned', dims=1:20, reduction.name = 'umap.aligned', reduction.key = 'alignedUMAP_')

# 5-3. Reduce dimension (ATAC)
DefaultAssay(cds.seurat) <- "ATAC"
cds.seurat <- RunTFIDF(cds.seurat)
cds.seurat <- FindTopFeatures(cds.seurat, min.cutoff = 20)
cds.seurat <- RunSVD(cds.seurat)
cds.seurat <- RunUMAP(cds.seurat, reduction = 'lsi', dims = 2:50, reduction.name = "umap.atac", reduction.key = "atacUMAP_")

# 5-4. WNN
cds.seurat <- FindMultiModalNeighbors(cds.seurat, reduction.list = list("Aligned", "lsi"), dims.list = list(1:20, 2:50))
cds.seurat <- RunUMAP(cds.seurat, nn.name = "weighted.nn", reduction.name = "wnn.umap", reduction.key = "wnnUMAP_")

# 5-5. Save
save(cds.seurat, file="Seurat/out/gex.atac.v2.RData")

# 5-6. Export coordinate
write.csv(x=cds.seurat[["wnn.umap"]][[,]], "Seurat/out/coordinate(wnn)_by_Seurat.v2.csv", quote=F)
"""

f = open(f'visualizing_clusters_wnn.r', 'w')
f.write(rscript3)
f.close()

In [ ]:
!Rscript visualizing_clusters_wnn.r

In [ ]:
# Fig1(b)_WNN_PLN.png
plot_cluster(
    "Seurat/PLN/out/coordinate(wnn)_by_Seurat.v2.csv",
    cdef,
    f'{Fig1b_WNN_PLN}.png',
    tissue="PLN",
    btype="Monocle",
    cxname="wnnUMAP_1",
    cyname="wnnUMAP_2",
    plot_width=700,
    plot_height=600,
    size=4
)

In [ ]:
# Fig1(b)_WNN_MLN.png
plot_cluster(
    "Seurat/MLN/out/coordinate(wnn)_by_Seurat.v2.csv",
    cdef,
    f'{Fig1b_WNN_MLN}.png',
    tissue="MLN",
    btype="Monocle",
    cxname="wnnUMAP_1",
    cyname="wnnUMAP_2",
    plot_width=700,
    plot_height=600,
    size=4
)

In [ ]:
# Fig1(b)_WNN_PP.png
plot_cluster(
    "Seurat/PP/out/coordinate(wnn)_by_Seurat.v2.csv",
    cdef,
    f'{Fig1b_WNN_PP}.png',
    tissue="PP",
    btype="Monocle",
    cxname="wnnUMAP_1",
    cyname="wnnUMAP_2",
    plot_width=700,
    plot_height=600,
    size=4
)

In [ ]:
# Fig1(b)_WNN_ALL.png
plot_cluster(
    "Seurat/out/coordinate(wnn)_by_Seurat.v2.csv",
    cdef,
    f'{Fig1b_WNN_ALL}.png',
    tissue="ALL",
    btype="Signac",
    cxname="wnnUMAP_1",
    cyname="wnnUMAP_2",
    plot_width=800,
    plot_height=600,
    size=4
)